# 🔬 Veri Madenciliği — Sınıflandırma: İleri Düzey Teknikler
## 2. Ders Uygulamalı Notebook
**Haydar Kılıç | Yapay Zeka Mühendisliği**

---
Bu notebook aşağıdaki konuları uygulamalı olarak ele almaktadır:

1. **K-En Yakın Komşu (KNN)**
2. **Naif Bayes Sınıflandırıcı (NBC)**
3. **Yapay Sinir Ağları (ANN) — Perseptron & Çok Katmanlı**
4. **Destek Vektör Makineleri (SVM)**

In [ ]:
# ========================================================
# GEREKLİ KÜTÜPHANELERİN KURULUMU VE İMPORT EDİLMESİ
# ========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# Sklearn modülleri
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import make_classification, make_circles, make_moons

print('✅ Tüm kütüphaneler başarıyla yüklendi!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

---
# BÖLÜM 1: K-En Yakın Komşu Sınıflandırıcı (KNN)

## 📖 Teorik Özet
KNN bir **tembel öğrenen (lazy learner)** algoritmasıdır. Eğitim sırasında model oluşturmak yerine tüm veriyi hafızada tutar. Tahmin yaparken:

$$y' = \arg\max_v \sum_{(x_i, y_i) \in D_z} I(v = y_i)$$

**Öklid Mesafesi:**
$$d(x, y) = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

**Mesafe Ağırlıklı Oylama:**
$$y' = \arg\max_v \sum_{(x_i, y_i) \in D_z} w_i \cdot I(v = y_i), \quad w_i = \frac{1}{d(x', x_i)^2}$$

## 1.1 Ders Slaytındaki Örneğin Adım Adım Uygulanması
**Problem:** Matematik ve Fizik notlarına göre öğrenci başarısını tahmin etme

In [ ]:
# ========================================================
# SLAYTTA VERİLEN ÖRNEĞİN SIFIRDAN UYGULAMASI
# ========================================================

# Eğitim verisi
data = {
    'Öğrenci': ['A', 'B', 'C', 'D', 'E'],
    'Matematik': [70, 60, 40, 30, 55],
    'Fizik':     [80, 65, 50, 35, 60],
    'Sınıf':     [1,   1, -1, -1,  1]   # 1: Başarılı, -1: Başarısız
}
df = pd.DataFrame(data)
print('=== EĞİTİM VERİSİ ===')
df_display = df.copy()
df_display['Sınıf'] = df_display['Sınıf'].map({1: '✓ Başarılı', -1: '✗ Başarısız'})
print(df_display.to_string(index=False))

# Test noktası
x_yeni = np.array([50, 55])
print(f'\n🎯 Yeni Öğrenci: Matematik={x_yeni[0]}, Fizik={x_yeni[1]}')

# Öklid mesafelerini hesapla
X_train = df[['Matematik', 'Fizik']].values
mesafeler = []
for i, row in df.iterrows():
    x = np.array([row['Matematik'], row['Fizik']])
    d = np.sqrt(np.sum((x_yeni - x)**2))
    mesafeler.append(d)

df['Mesafe'] = mesafeler
df_sorted = df.sort_values('Mesafe').reset_index(drop=True)

print('\n=== MESAFELERİ KÜÇÜKTEN BÜYÜĞE SIRALAMA ===')
df_sorted_display = df_sorted.copy()
df_sorted_display['Sınıf'] = df_sorted_display['Sınıf'].map({1: '✓', -1: '✗'})
df_sorted_display['Mesafe'] = df_sorted_display['Mesafe'].round(2)
print(df_sorted_display[['Öğrenci', 'Matematik', 'Fizik', 'Mesafe', 'Sınıf']].to_string(index=False))

In [ ]:
# ========================================================
# k=3 İÇİN SINIFLANDIRMA KARARI
# ========================================================
k = 3
k_komsular = df_sorted.head(k)

print(f'=== k={k} EN YAKIN KOMŞU ===')
for _, row in k_komsular.iterrows():
    sinif_str = '✓ Başarılı' if row['Sınıf'] == 1 else '✗ Başarısız'
    print(f"  {row['Öğrenci']}: d={row['Mesafe']:.2f} → {sinif_str}")

# Çoğunluk oylaması
oylar = k_komsular['Sınıf'].values
basarili = np.sum(oylar == 1)
basarisiz = np.sum(oylar == -1)
print(f'\n📊 Oy Sayımı: ✓ Başarılı = {basarili} | ✗ Başarısız = {basarisiz}')
tahmin = '✓ BAŞARILI' if basarili > basarisiz else '✗ BAŞARISIZ'
print(f'🏆 Karar: Yeni öğrenci → {tahmin}')

print('\n=== MESAFE AĞIRLIKLI OYLAMA (Opsiyonel) ===')
toplam_agirlik = {1: 0.0, -1: 0.0}
for _, row in k_komsular.iterrows():
    w = 1.0 / row['Mesafe']
    toplam_agirlik[row['Sınıf']] += w
    sinif_str = '✓' if row['Sınıf'] == 1 else '✗'
    print(f"  {row['Öğrenci']}: w = 1/{row['Mesafe']:.2f} = {w:.4f} → {sinif_str}")

print(f'\n  S(✓) = {toplam_agirlik[1]:.4f}')
print(f'  S(✗) = {toplam_agirlik[-1]:.4f}')
agirlikli_tahmin = '✓ BAŞARILI' if toplam_agirlik[1] > toplam_agirlik[-1] else '✗ BAŞARISIZ'
print(f'🏆 Ağırlıklı Karar: Yeni öğrenci → {agirlikli_tahmin}')

In [ ]:
# ========================================================
# GÖRSELLEŞTİRME: k'nın ETKİSİ
# ========================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('KNN — k Değerinin Karar Sınırına Etkisi', fontsize=14, fontweight='bold')

renkler = {1: '#2ecc71', -1: '#e74c3c'}
etiketler = {1: '✓ Başarılı', -1: '✗ Başarısız'}

for ax_idx, k_val in enumerate([1, 3, 5]):
    ax = axes[ax_idx]
    
    # Karar ızgarası
    xx, yy = np.meshgrid(np.linspace(20, 85, 200), np.linspace(25, 90, 200))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    # Manuel KNN
    def knn_predict_manual(test_point, X, y, k):
        dists = np.sqrt(np.sum((X - test_point)**2, axis=1))
        idx = np.argsort(dists)[:k]
        votes = y[idx]
        return 1 if np.sum(votes == 1) >= np.sum(votes == -1) else -1
    
    X_tr = df[['Matematik', 'Fizik']].values
    y_tr = df['Sınıf'].values
    
    Z = np.array([knn_predict_manual(p, X_tr, y_tr, k_val) for p in grid_points])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.8, linestyles='--')
    
    # Eğitim noktaları
    for _, row in df.iterrows():
        ax.scatter(row['Matematik'], row['Fizik'], 
                   c=renkler[row['Sınıf']], s=120, edgecolors='black', zorder=5)
        ax.annotate(row['Öğrenci'], (row['Matematik']+0.5, row['Fizik']+0.5), fontsize=9)
    
    # Test noktası
    ax.scatter(*x_yeni, c='blue', marker='*', s=250, zorder=6, label='Yeni Öğrenci')
    
    # k komşu çemberi
    dists_sorted = sorted([(np.sqrt(np.sum((x_yeni - X_tr[i])**2)), i) for i in range(len(X_tr))])
    r = dists_sorted[k_val-1][0] + 0.5
    circle = plt.Circle(x_yeni, r, color='blue', fill=False, linestyle=':', linewidth=1.5)
    ax.add_patch(circle)
    
    ax.set_title(f'k = {k_val}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Matematik Notu')
    ax.set_ylabel('Fizik Notu')
    # legend after patches
    green_patch = mpatches.Patch(color='#2ecc71', label='Başarılı')
    red_patch = mpatches.Patch(color='#e74c3c', label='Başarısız')
    ax.legend(handles=[green_patch, red_patch], loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()
print('💡 k küçük → esnek karar sınırı (overfitting riski)')
print('💡 k büyük → yumuşak karar sınırı (underfitting riski)')

## 1.2 Sklearn ile KNN — Gerçek Veri Üzerinde

In [ ]:
# ========================================================
# SKLEARN KNN — IRIS VERİ SETİ
# ========================================================
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data[:, :2], iris.target  # İlk 2 özellik (görselleştirme için)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Farklı k değerleri için doğruluk
k_degerler = range(1, 21)
dogruluklar = []

for k in k_degerler:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    dogruluklar.append(acc)

# En iyi k
en_iyi_k = k_degerler[np.argmax(dogruluklar)]
print(f'✅ En iyi k = {en_iyi_k} | Doğruluk = {max(dogruluklar):.2%}')

# Grafik
plt.figure(figsize=(10, 4))
plt.plot(k_degerler, dogruluklar, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=en_iyi_k, color='red', linestyle='--', label=f'En iyi k={en_iyi_k}')
plt.xlabel('k Değeri', fontsize=12)
plt.ylabel('Test Doğruluğu', fontsize=12)
plt.title('KNN: k Parametresinin Doğruluğa Etkisi (Iris Veri Seti)', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# En iyi modelin raporu
knn_best = KNeighborsClassifier(n_neighbors=en_iyi_k)
knn_best.fit(X_train, y_train)
y_pred = knn_best.predict(X_test)
print('\n=== SINIFLANDIRMA RAPORU ===')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

---
# BÖLÜM 2: Naif Bayes Sınıflandırıcı (NBC)

## 📖 Teorik Özet
Naif Bayes, özniteliklerin **koşullu bağımsızlığı** varsayımına dayanır:

$$P(x|y) = \prod_{i=1}^{d} P(x_i|y)$$

Sonsal olasılık (Bayes Teoremi):

$$P(y|x) \propto P(y) \prod_{i=1}^{d} P(x_i|y)$$

**Karar:** En yüksek sonsal olasılığa sahip sınıfı seç.

In [ ]:
# ========================================================
# SLAYTTAKI NBC ÖRNEĞİNİN UYGULAMASI
# Temerrüde Düşen Borçlu Tahmini
# ========================================================

# Eğitim verisi
nbc_data = pd.DataFrame({
    'Tid': range(1, 11),
    'HomeOwner': ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'MaritalStatus': ['Single','Married','Single','Married','Divorced','Married','Divorced','Single','Married','Single'],
    'AnnualIncome': [125, 100, 70, 120, 95, 60, 220, 85, 75, 90],
    'Defaulted': ['No','No','No','No','Yes','No','No','Yes','No','Yes']
})

print('=== EĞİTİM VERİSİ ===')
print(nbc_data.to_string(index=False))

# Önsel olasılıklar
P_evet = (nbc_data['Defaulted'] == 'Yes').mean()
P_hayir = (nbc_data['Defaulted'] == 'No').mean()
print(f'\n📊 Önsel Olasılıklar:')
print(f'  P(Evet)  = {P_evet:.1f}')
print(f'  P(Hayır) = {P_hayir:.1f}')

In [ ]:
# ========================================================
# SINIFLARA GÖRE KOŞULLU OLASILIKLAR
# ========================================================
evet_df = nbc_data[nbc_data['Defaulted'] == 'Yes']
hayir_df = nbc_data[nbc_data['Defaulted'] == 'No']

print('=== "EVET" SINIFI (Temerrüde Düşenler) ===')
print(evet_df[['HomeOwner','MaritalStatus','AnnualIncome']].to_string(index=False))

# Kategorik özellikler için P(xi|y)
def kosullu_olasilik(df_sinif, kolon, deger):
    return (df_sinif[kolon] == deger).sum() / len(df_sinif)

# Test örneği: HomeOwner=No, Married, Income=120K
print('\n=== TEST ÖRNEĞİ: HomeOwner=No, Married, Gelir=120K ===')

# Hayır sınıfı için
P_ho_no_hayir  = kosullu_olasilik(hayir_df, 'HomeOwner', 'No')
P_ms_m_hayir   = kosullu_olasilik(hayir_df, 'MaritalStatus', 'Married')

# Sürekli nitelik için Gaussian
mu_hayir = hayir_df['AnnualIncome'].mean()
std_hayir = hayir_df['AnnualIncome'].std(ddof=1)
from scipy.stats import norm
P_inc_hayir = norm.pdf(120, mu_hayir, std_hayir)

P_x_hayir = P_ho_no_hayir * P_ms_m_hayir * P_inc_hayir
print(f'\n"Hayır" Sınıfı:')
print(f'  P(HomeOwner=No | Hayır)       = {P_ho_no_hayir:.4f}  [{int(P_ho_no_hayir*len(hayir_df))}/{len(hayir_df)}]')
print(f'  P(Married | Hayır)             = {P_ms_m_hayir:.4f}  [{int(P_ms_m_hayir*len(hayir_df))}/{len(hayir_df)}]')
print(f'  P(Gelir=120K | Hayır) [Gauss]  = {P_inc_hayir:.6f}  (μ={mu_hayir:.1f}, σ={std_hayir:.1f})')
print(f'  ──────────────────────────────────')
print(f'  P(x|Hayır) = {P_x_hayir:.8f}')

# Evet sınıfı için
P_ho_no_evet  = kosullu_olasilik(evet_df, 'HomeOwner', 'No')
P_ms_m_evet   = kosullu_olasilik(evet_df, 'MaritalStatus', 'Married')
mu_evet = evet_df['AnnualIncome'].mean()
std_evet = evet_df['AnnualIncome'].std(ddof=1)
P_inc_evet = norm.pdf(120, mu_evet, std_evet)
P_x_evet = P_ho_no_evet * P_ms_m_evet * P_inc_evet

print(f'\n"Evet" Sınıfı:')
print(f'  P(HomeOwner=No | Evet)         = {P_ho_no_evet:.4f}')
print(f'  P(Married | Evet)              = {P_ms_m_evet:.4f}  ← Sıfır Olasılık!')
print(f'  P(Gelir=120K | Evet) [Gauss]   = {P_inc_evet:.2e}')
print(f'  P(x|Evet) = {P_x_evet:.8f}')

print('\n=== SONSAL OLASILIKLAR (Normalleştirilmemiş) ===')
skor_hayir = P_hayir * P_x_hayir
skor_evet  = P_evet  * P_x_evet
print(f'  P(Hayır) × P(x|Hayır) = {skor_hayir:.8f}')
print(f'  P(Evet)  × P(x|Evet)  = {skor_evet:.8f}')
karar = 'Hayır (Temerrüde Düşmez)' if skor_hayir > skor_evet else 'Evet (Temerrüde Düşer)'
print(f'\n🏆 KARAR: {karar}')

In [ ]:
# ========================================================
# SKLEARN GaussianNB — KARŞILAŞTIRMA
# ========================================================
from sklearn.preprocessing import LabelEncoder

# Kodlama
le_ho = LabelEncoder(); le_ms = LabelEncoder(); le_d = LabelEncoder()
X_nbc = np.column_stack([
    le_ho.fit_transform(nbc_data['HomeOwner']),
    le_ms.fit_transform(nbc_data['MaritalStatus']),
    nbc_data['AnnualIncome'].values
])
y_nbc = le_d.fit_transform(nbc_data['Defaulted'])

gnb = GaussianNB()
gnb.fit(X_nbc, y_nbc)

x_test_enc = np.array([[
    le_ho.transform(['No'])[0],
    le_ms.transform(['Married'])[0],
    120
]])

pred = gnb.predict(x_test_enc)[0]
proba = gnb.predict_proba(x_test_enc)[0]

print('=== sklearn GaussianNB ile Doğrulama ===')
print(f'Tahmin Edilen Sınıf : {le_d.inverse_transform([pred])[0]}')
print(f'P(No)  = {proba[le_d.transform(["No"])[0]]:.6f}')
print(f'P(Yes) = {proba[le_d.transform(["Yes"])[0]]:.6f}')
print('\n✅ Manuel hesaplamamızla aynı sonuç!')

---
# BÖLÜM 3: Yapay Sinir Ağları (ANN)

## 📖 Teorik Özet

### Perseptron
$$\hat{y} = \text{sign}(\tilde{w}^T \tilde{x}) = \begin{cases} 1 & \text{eğer } w^Tx + b > 0 \\ -1 & \text{aksi halde} \end{cases}$$

### Ağırlık Güncelleme Kuralı
$$w_j^{(k+1)} = w_j^{(k)} + \lambda(y_i - \hat{y}_i^{(k)}) x_{ij}$$

### Çok Katmanlı Ağ — Aktivasyon
$$a_i^l = f(z_i^l) = f\left(\sum_j w_{ij}^l a_j^{l-1} + b_i^l\right)$$

### Sigmoid Fonksiyonu
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
# ========================================================
# AKTİVASYON FONKSİYONLARI GÖRSELLEŞTİRME
# ========================================================
z = np.linspace(-4, 4, 300)

aktivasyonlar = {
    'Linear': z,
    'Sigmoid (σ)': 1 / (1 + np.exp(-z)),
    'Tanh': np.tanh(z),
    'Sign': np.sign(z),
    'ReLU': np.maximum(0, z)
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Aktivasyon Fonksiyonları', fontsize=14, fontweight='bold')

renkler = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6', '#e74c3c']

for ax, (isim, deger), renk in zip(axes, aktivasyonlar.items(), renkler):
    ax.plot(z, deger, color=renk, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title(isim, fontweight='bold', fontsize=11)
    ax.set_xlim(-4, 4)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('z')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# SIFIRDAN PERSEPTRON UYGULAMASI
# ========================================================
class Perceptron:
    """Slaytta anlatılan Perceptron öğrenme algoritmasının sıfırdan uygulaması"""
    
    def __init__(self, learning_rate=0.1, max_iter=100, threshold=0.01):
        self.lr = learning_rate
        self.max_iter = max_iter
        self.threshold = threshold
        self.weights = None
        self.bias = None
        self.hatalar = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Rastgele başlangıç ağırlıkları
        np.random.seed(42)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
        for epoch in range(self.max_iter):
            toplam_hata = 0
            for xi, yi in zip(X, y):
                y_hat = self._predict_one(xi)
                hata = yi - y_hat
                # Ağırlık güncelleme: w(k+1) = w(k) + λ(y - ŷ)x
                self.weights += self.lr * hata * xi
                self.bias    += self.lr * hata
                toplam_hata  += abs(hata)
            
            ort_hata = toplam_hata / n_samples
            self.hatalar.append(ort_hata)
            
            if ort_hata < self.threshold:
                print(f'  ✅ Yakınsama sağlandı! Epoch={epoch+1}, Ortalama Hata={ort_hata:.4f}')
                break
        
        return self
    
    def _predict_one(self, x):
        z = np.dot(self.weights, x) + self.bias
        return 1 if z > 0 else -1
    
    def predict(self, X):
        return np.array([self._predict_one(xi) for xi in X])


# AND kapısı örneği (doğrusal ayrılabilir)
X_and = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([-1, -1, -1, 1])  # AND: sadece (1,1) için True

print('=== PERSEPTRON — AND KAPISI ===')
print('Veriler:', list(zip(X_and.tolist(), y_and.tolist())))

p = Perceptron(learning_rate=0.1, max_iter=50)
p.fit(X_and, y_and)

tahminler = p.predict(X_and)
print(f'Tahminler : {tahminler}')
print(f'Gerçekler : {y_and}')
print(f'Doğruluk  : {accuracy_score(y_and, tahminler):.2%}')

# Hata grafiği
plt.figure(figsize=(8, 3))
plt.plot(p.hatalar, 'b-o', markersize=6)
plt.xlabel('Epoch')
plt.ylabel('Ortalama Hata')
plt.title('Perseptron Öğrenme Eğrisi — AND Kapısı')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# PERSEPTRONUN KISITLARI — XOR PROBLEMI
# (Doğrusal Ayrılamaz → Başarısız)
# ========================================================
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1])  # XOR: doğrusal ayrılamaz!

print('=== PERSEPTRONUN KISITLARI — XOR PROBLEMI ===')
p_xor = Perceptron(learning_rate=0.1, max_iter=100, threshold=0.001)
p_xor.fit(X_xor, y_xor)

tahminler_xor = p_xor.predict(X_xor)
print(f'Tahminler: {tahminler_xor}')
print(f'Gerçekler: {y_xor}')
print(f'Doğruluk : {accuracy_score(y_xor, tahminler_xor):.2%}')
print('\n⚠️ Perseptron XOR problemini çözemiyor → Çok Katmanlı Ağ gerekli!')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
problemler = [('AND (Doğrusal Ayrılabilir ✅)', X_and, y_and, p),
              ('XOR (Doğrusal Ayrılamaz ❌)', X_xor, y_xor, p_xor)]

for ax, (baslik, X, y, model) in zip(axes, problemler):
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
    ax.contour(xx, yy, Z, colors='black', linewidths=1.5)
    
    for xi, yi in zip(X, y):
        c = 'green' if yi == 1 else 'red'
        m = '^' if yi == 1 else 'o'
        ax.scatter(*xi, c=c, marker=m, s=150, edgecolors='black', zorder=5)
    
    ax.set_title(baslik, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# ÇOK KATMANLI SİNİR AĞI — sklearn MLP
# ========================================================
print('=== ÇOK KATMANLI SİNİR AĞI (MLP) — XOR Çözümü ===')

mlp_xor = MLPClassifier(
    hidden_layer_sizes=(4,),   # 1 gizli katman, 4 nöron
    activation='relu',
    max_iter=1000,
    random_state=42
)
mlp_xor.fit(X_xor, y_xor)
print(f'XOR Doğruluğu: {accuracy_score(y_xor, mlp_xor.predict(X_xor)):.2%}')
print('✅ Çok katmanlı ağ XOR problemini çözebiliyor!')

# Gerçek sınıflandırma problemi
print('\n=== MLP — IRIS VERİ SETİ ===')
iris_full = load_iris()
X_full, y_full = iris_full.data, iris_full.target
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

mimariler = [
    (50,),
    (100, 50),
    (100, 50, 25)
]

print("{:<25} {:<22} {}".format("Mimari", "Eğitim Doğruluğu", "Test Doğruluğu"))
print('-' * 60)
for mimari in mimariler:
    mlp = MLPClassifier(hidden_layer_sizes=mimari, max_iter=500, random_state=42)
    mlp.fit(X_tr_s, y_tr)
    train_acc = accuracy_score(y_tr, mlp.predict(X_tr_s))
    test_acc  = accuracy_score(y_te, mlp.predict(X_te_s))
    print(f"{str(mimari):<25} {train_acc:.2%}               {test_acc:.2%}")

In [ ]:
# ========================================================
# DERİN SİNİR AĞLARI — VANISHING GRADIENT VİZÜELİ
# ========================================================
z_vals = np.linspace(-6, 6, 300)
sigmoid = 1 / (1 + np.exp(-z_vals))
sigmoid_grad = sigmoid * (1 - sigmoid)  # σ'(z) = σ(z)(1-σ(z))
relu = np.maximum(0, z_vals)
relu_grad = (z_vals > 0).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Vanishing Gradient Problemi: Sigmoid vs ReLU', fontsize=13, fontweight='bold')

# Sigmoid
ax1 = axes[0]
ax1.plot(z_vals, sigmoid, 'b-', label='σ(z)', linewidth=2)
ax1.plot(z_vals, sigmoid_grad, 'r--', label="σ'(z) — Gradyan", linewidth=2)
ax1.axhline(0.25, color='gray', linestyle=':', alpha=0.7, label='Maks. Gradyan = 0.25')
ax1.fill_between(z_vals, sigmoid_grad, alpha=0.2, color='red')
ax1.set_title('Sigmoid: Gradyan Yok Oluyor ⚠️', fontweight='bold')
ax1.set_xlabel('z'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.annotate('Doygunluk bölgesi\n(gradient ≈ 0)', xy=(4, 0.02), fontsize=9,
             color='red', ha='center')

# ReLU
ax2 = axes[1]
ax2.plot(z_vals, relu, 'b-', label='ReLU(z)', linewidth=2)
ax2.plot(z_vals, relu_grad, 'r--', label="ReLU'(z) — Gradyan", linewidth=2)
ax2.set_title('ReLU: Gradyan Korunuyor ✅', fontweight='bold')
ax2.set_xlabel('z'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 Sigmoid: Gradyan maksimum 0.25 — derin ağlarda çarpımla yok olur')
print('💡 ReLU: Pozitif bölgede gradyan = 1 — gradyan akışı korunur')

---
# BÖLÜM 4: Destek Vektör Makineleri (SVM)

## 📖 Teorik Özet

### Hiperdüzlem
$$w^T x + b = 0$$

### Marj Maksimizasyonu (Primal Problem)
$$\min_{w,b} \frac{\|w\|^2}{2} \quad \text{s.t.} \quad y_i(w^T x_i + b) \geq 1$$

### Dual Problem
$$\max_{\lambda} \sum_{i=1}^n \lambda_i - \frac{1}{2}\sum_{i,j} \lambda_i \lambda_j y_i y_j x_i^T x_j$$

### Kernel Fonksiyonları
- **Polinom:** $K(u,v) = (u^T v + 1)^p$
- **RBF:** $K(u,v) = e^{-\|u-v\|^2 / (2\sigma^2)}$
- **Sigmoid:** $K(u,v) = \tanh(ku^Tv - \delta)$

In [ ]:
# ========================================================
# SLAYTTAKI LİNEER SVM ÖRNEĞİ — SCIPY İLE DOĞRULAMA
# ========================================================
# Slaytta verilen 8 örnek
X_svm = np.array([
    [0.3858, 0.4687],
    [0.4871, 0.6110],
    [0.9218, 0.4103],
    [0.7382, 0.8936],
    [0.1763, 0.0579],
    [0.4057, 0.3529],
    [0.9355, 0.8132],
    [0.2146, 0.0099]
])
y_svm = np.array([1, -1, -1, -1, 1, 1, -1, 1])

# sklearn SVM ile çöz
svm_lin = SVC(kernel='linear', C=1e6)  # Büyük C → sert marj
svm_lin.fit(X_svm, y_svm)

w = svm_lin.coef_[0]
b = svm_lin.intercept_[0]

print('=== LİNEER SVM SONUÇLARI ===')
print(f'Ağırlık vektörü  w = [{w[0]:.4f}, {w[1]:.4f}]')
print(f'Sapma terimi     b = {b:.4f}')
print(f'Karar sınırı: {w[0]:.2f}·x₁ + {w[1]:.2f}·x₂ + {b:.2f} = 0')
print(f'\nDestek Vektörleri:')
for sv in svm_lin.support_vectors_:
    print(f'  {sv}')
print(f'\nSlaytta: -6.64·x₁ - 9.32·x₂ + 7.93 = 0')
print('✅ Sonuçlar uyuşuyor!')

In [ ]:
# ========================================================
# LİNEER SVM GÖRSELLEŞTİRME — MARJ VE DESTEK VEKTÖRLER
# ========================================================
fig, ax = plt.subplots(figsize=(8, 6))

# Karar sınırı ve marjlar
x1_range = np.linspace(-0.1, 1.1, 200)

def karar_siniri(x1, w, b, offset=0):
    return (-w[0]*x1 - b + offset) / w[1]

ax.plot(x1_range, karar_siniri(x1_range, w, b), 'k-', linewidth=2.5, label='Karar Sınırı')
ax.plot(x1_range, karar_siniri(x1_range, w, b, 1), 'b--', linewidth=1.5, label='Marj (+1)')
ax.plot(x1_range, karar_siniri(x1_range, w, b, -1), 'r--', linewidth=1.5, label='Marj (-1)')

# Marj bölgesi
y_up = karar_siniri(x1_range, w, b, 1)
y_down = karar_siniri(x1_range, w, b, -1)
ax.fill_between(x1_range, y_down, y_up, alpha=0.1, color='yellow', label='Marj Bölgesi')

# Veri noktaları
for xi, yi in zip(X_svm, y_svm):
    c = '#2ecc71' if yi == 1 else '#e74c3c'
    m = 's' if yi == 1 else 'o'
    ax.scatter(*xi, c=c, marker=m, s=100, edgecolors='black', zorder=5)

# Destek vektörlerini işaretle
for sv in svm_lin.support_vectors_:
    circle = plt.Circle(sv, 0.035, color='blue', fill=False, linewidth=2.5, zorder=6)
    ax.add_patch(circle)

ax.set_xlim(-0.1, 1.2); ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('x₁', fontsize=12); ax.set_ylabel('x₂', fontsize=12)
ax.set_title('Lineer SVM: Maksimum Marjlı Hiperdüzlem', fontsize=13, fontweight='bold')

green_patch = mpatches.Patch(color='#2ecc71', label='y = +1')
red_patch   = mpatches.Patch(color='#e74c3c', label='y = -1')
ax.legend(handles=[green_patch, red_patch,
          plt.Line2D([],[], color='k', label='Karar Sınırı'),
          plt.Line2D([],[], color='b', linestyle='--', label='Marj ±1')],
          loc='upper left')

ax.text(0.5, 0.02, f'{w[0]:.2f}·x₁ + {w[1]:.2f}·x₂ + {b:.2f} = 0',
        fontsize=10, ha='center', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.show()
print('🔵 Daire içindeki noktalar = Destek Vektörler')

In [ ]:
# ========================================================
# DOĞRUSAL OLMAYAN SVM — KERNEL FONKSİYONLARI
# ========================================================
# Slaytta gösterilen: daireler merkezde, kareler dışta
np.random.seed(42)
X_circles, y_circles = make_circles(n_samples=100, noise=0.1, factor=0.4, random_state=42)
y_circles_svm = np.where(y_circles == 0, -1, 1)

kerneller = {
    'Linear (Doğrusal)': SVC(kernel='linear'),
    'Poly (p=3)':         SVC(kernel='poly', degree=3),
    'RBF (Radyal)':       SVC(kernel='rbf', gamma=1.0),
    'Sigmoid':            SVC(kernel='sigmoid')
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('SVM Kernel Karşılaştırması — Daire Veri Seti', fontsize=13, fontweight='bold')

for ax, (isim, model) in zip(axes, kerneller.items()):
    model.fit(X_circles, y_circles_svm)
    acc = accuracy_score(y_circles_svm, model.predict(X_circles))
    
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
    ax.contour(xx, yy, Z, colors='black', linewidths=0.8)
    
    for xi, yi in zip(X_circles, y_circles_svm):
        c = '#27ae60' if yi == 1 else '#c0392b'
        ax.scatter(*xi, c=c, s=30, edgecolors='none', alpha=0.8)
    
    ax.set_title(f'{isim}\nDoğruluk: {acc:.2%}', fontsize=10, fontweight='bold')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

---
# BÖLÜM 5: Kapsamlı Karşılaştırma
## Tüm Algoritmaların Aynı Veri Seti Üzerinde Değerlendirilmesi

In [ ]:
# ========================================================
# TÜM ALGORİTMALARIN KARŞILAŞTIRMASI
# ========================================================
from sklearn.datasets import load_breast_cancer

veri = load_breast_cancer()
X_c, y_c = veri.data, veri.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

modeller = {
    'KNN (k=5)':          KNeighborsClassifier(n_neighbors=5),
    'KNN (k=11)':         KNeighborsClassifier(n_neighbors=11),
    'Naif Bayes':         GaussianNB(),
    'MLP (1 gizli kat.)': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
    'MLP (2 gizli kat.)': MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500, random_state=42),
    'SVM (Lineer)':       SVC(kernel='linear'),
    'SVM (RBF)':          SVC(kernel='rbf')
}

sonuclar = []
print("{:<25} {:<20} {}".format("Model", "Eğitim Doğruluk", "Test Doğruluk"))
print('=' * 60)

for isim, model in modeller.items():
    model.fit(X_tr_s, y_tr)
    train_acc = accuracy_score(y_tr, model.predict(X_tr_s))
    test_acc  = accuracy_score(y_te, model.predict(X_te_s))
    sonuclar.append({'Model': isim, 'Eğitim': train_acc, 'Test': test_acc})
    print(f"{isim:<25} {train_acc:.4f}              {test_acc:.4f}")

# Görsel karşılaştırma
df_sonuc = pd.DataFrame(sonuclar)

x = np.arange(len(df_sonuc))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, df_sonuc['Eğitim'], width, label='Eğitim', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, df_sonuc['Test'],   width, label='Test',   color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_sonuc['Model'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Doğruluk', fontsize=12)
ax.set_title('Tüm Sınıflandırıcıların Karşılaştırması (Breast Cancer Veri Seti)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.85, 1.02)
ax.grid(True, axis='y', alpha=0.4)

for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# KARAR SINIRLARI KARŞILAŞTIRMASI (2D)
# ========================================================
X_2d, y_2d = make_moons(n_samples=200, noise=0.25, random_state=42)
X_2d_tr, X_2d_te, y_2d_tr, y_2d_te = train_test_split(X_2d, y_2d, test_size=0.2, random_state=42)

modeller_2d = {
    'KNN (k=3)':      KNeighborsClassifier(n_neighbors=3),
    'Naif Bayes':     GaussianNB(),
    'MLP':            MLPClassifier(hidden_layer_sizes=(50,25), max_iter=500, random_state=42),
    'SVM (RBF)':      SVC(kernel='rbf', probability=True)
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Karar Sınırları Karşılaştırması — Half-Moon Veri Seti', fontsize=13, fontweight='bold')

xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2.5, 200))

for ax, (isim, model) in zip(axes, modeller_2d.items()):
    model.fit(X_2d_tr, y_2d_tr)
    test_acc = accuracy_score(y_2d_te, model.predict(X_2d_te))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=ListedColormap(['#aed6f1', '#a9dfbf']))
    ax.contour(xx, yy, Z, colors='black', linewidths=1)
    
    for xi, yi in zip(X_2d, y_2d):
        c = '#2980b9' if yi == 0 else '#27ae60'
        ax.scatter(*xi, c=c, s=25, alpha=0.7)
    
    ax.set_title(f'{isim}\nTest Doğruluk: {test_acc:.2%}', fontsize=10, fontweight='bold')
    ax.set_xlim(-2, 3); ax.set_ylim(-1.5, 2.5)

plt.tight_layout()
plt.show()

---
# 📝 ÖZET TABLOSU

| Algoritma | Tip | Avantaj | Dezavantaj | En İyi Kullanım |
|-----------|-----|---------|------------|----------------|
| **KNN** | Tembel öğrenen | Basit, yorumlanabilir | Yüksek bellek/zaman | Küçük-orta veri |
| **Naif Bayes** | Olasılıksal | Hızlı, eksik veriyi işler | Bağımsızlık varsayımı | Metin sınıflandırma |
| **ANN/MLP** | Sinir ağı | Doğrusal olmayan sınırlar | Aşırı öğrenme riski | Büyük karmaşık veri |
| **SVM** | Marj tabanlı | İyi genelleme, kernel trick | Büyük veride yavaş | Yüksek boyutlu veri |

## 🔑 Temel Kavramlar
- **İstekli öğrenen** (eager learner): Model eğitim sırasında oluşturulur — Karar ağaçları, kural tabanlı, ANN, SVM
- **Tembel öğrenen** (lazy learner): Tahmin anında sınıflandırma yapılır — KNN
- **Overfitting**: Eğitim verisi ezber → küçük k (KNN), fazla katman (ANN)
- **Underfitting**: Model yetersiz → büyük k (KNN), az nöron (ANN)
- **Kernel trick**: Doğrusal olmayan verileri yüksek boyuta taşıyarak lineer SVM uygulamak